In [1]:
!nvidia-smi

Sat May  9 21:56:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate
!pip install -q huggingface_hub sentencepiece protobuf
print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00
✅ All packages installed!


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset

# Load Medical Meadow dataset - medical QA pairs
dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards")

print(dataset)
print(f"\nTotal examples: {len(dataset['train'])}")
print(f"\nSample example:")
print(dataset['train'][0])

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 33955
    })
})

Total examples: 33955

Sample example:
{'input': 'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?', 'output': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.', 'instruction': 'Answer this question truthfully'}


In [5]:
# Format into instruction template
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

# Apply formatting
formatted = dataset['train'].map(format_prompt)

# Split into train and validation
split = formatted.train_test_split(test_size=0.1, seed=42)
train_data = split['train']
val_data   = split['test']

print(f"✅ Train size : {len(train_data)}")
print(f"✅ Val size   : {len(val_data)}")
print(f"\nSample formatted prompt:")
print(train_data[0]['text'])

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

✅ Train size : 30559
✅ Val size   : 3396

Sample formatted prompt:
### Instruction:
Answer this question truthfully

### Input:
How do ergot alkaloids work, and what are they used for?

### Response:
The mechanism of action of ergot alkaloids is that they are serotonin (5-HT) agonists. Ergot alkaloids are a group of medications that are used to treat various conditions, including migraines, cluster headaches, and postpartum hemorrhage. They work by binding to and activating serotonin receptors in the brain and blood vessels, which can lead to vasoconstriction and a reduction in inflammation. The exact mechanism by which ergot alkaloids relieve migraines is not fully understood, but it is thought to be related to their ability to constrict blood vessels in the brain and reduce the release of inflammatory substances. Ergot alkaloids are available in various forms, including tablets, injections, and nasal sprays. They are generally considered to be effective treatments for migraines and o

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mistral — no gating, works immediately, excellent for medical QA
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit... (3-5 mins)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

total = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded!")
print(f"Total parameters : {total:,}")
print(f"Device           : {model.device}")

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


✅ Model loaded!
Total parameters : 3,758,362,624
Device           : cuda:0


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for LoRA training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                      # LoRA rank
    lora_alpha=32,             # LoRA alpha (scaling)
    target_modules=[           # Which layers to apply LoRA to
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct              = 100 * trainable_params / total_params

print(f"✅ LoRA applied!")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Trainable %          : {pct:.4f}%")
print(f"\nLoRA rank     : {lora_config.r}")
print(f"LoRA alpha    : {lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

✅ LoRA applied!
Total parameters     : 3,771,994,112
Trainable parameters : 13,631,488
Trainable %          : 0.3614%

LoRA rank     : 16
LoRA alpha    : 32
Target modules: {'k_proj', 'v_proj', 'q_proj', 'o_proj'}


In [12]:
# Run this first to see exact SFTConfig parameters
from trl import SFTConfig
import inspect
params = inspect.signature(SFTConfig.__init__).parameters
print("SFTConfig parameters:")
for p in params:
    print(f"  {p}")

SFTConfig parameters:
  self
  output_dir
  do_train
  do_eval
  do_predict
  eval_strategy
  prediction_loss_only
  per_device_train_batch_size
  per_device_eval_batch_size
  gradient_accumulation_steps
  eval_accumulation_steps
  eval_delay
  torch_empty_cache_steps
  learning_rate
  weight_decay
  adam_beta1
  adam_beta2
  adam_epsilon
  max_grad_norm
  num_train_epochs
  max_steps
  lr_scheduler_type
  lr_scheduler_kwargs
  warmup_ratio
  warmup_steps
  log_level
  log_level_replica
  log_on_each_node
  logging_dir
  logging_strategy
  logging_first_step
  logging_steps
  logging_nan_inf_filter
  save_strategy
  save_steps
  save_total_limit
  enable_jit_checkpoint
  save_on_each_node
  save_only_model
  restore_callback_states_from_checkpoint
  use_cpu
  seed
  data_seed
  bf16
  fp16
  bf16_full_eval
  fp16_full_eval
  tf32
  local_rank
  ddp_backend
  debug
  dataloader_drop_last
  eval_steps
  dataloader_num_workers
  dataloader_prefetch_factor
  run_name
  disable_tqdm
  remov

In [13]:
from trl import SFTTrainer, SFTConfig

small_train = train_data.select(range(5000))
small_val   = val_data.select(range(500))

print(f"Training on  : {len(small_train)} examples")
print(f"Validating on: {len(small_val)} examples")

training_args = SFTConfig(
    output_dir                  = "./medical-mistral-lora",
    num_train_epochs            = 1,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    weight_decay                = 0.001,
    warmup_steps                = 50,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    load_best_model_at_end      = True,
    logging_steps               = 25,
    fp16                        = True,
    dataset_text_field          = "text",
    report_to                   = "none",
    max_length                  = 512,
)

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = small_train,
    eval_dataset     = small_val,
    processing_class = tokenizer,
)

print("✅ Trainer ready!")
print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Max sequence length  : {training_args.max_length}")
print(f"Learning rate        : {training_args.learning_rate}")

Training on  : 5000 examples
Validating on: 500 examples


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Trainer ready!
Effective batch size : 16
Max sequence length  : 512
Learning rate        : 0.0002


In [15]:
from trl import SFTTrainer, SFTConfig

small_train = train_data.select(range(5000))
small_val   = val_data.select(range(500))

print(f"Training on  : {len(small_train)} examples")
print(f"Validating on: {len(small_val)} examples")

training_args = SFTConfig(
    output_dir                  = "./medical-mistral-lora",
    num_train_epochs            = 1,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    weight_decay                = 0.001,
    warmup_steps                = 50,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    load_best_model_at_end      = True,
    logging_steps               = 25,
    fp16                        = False,   # ← disabled
    bf16                        = False,   # ← disabled
    dataset_text_field          = "text",
    report_to                   = "none",
    max_length                  = 512,
    optim                       = "adamw_torch",
)

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = small_train,
    eval_dataset     = small_val,
    processing_class = tokenizer,
)

print("✅ Trainer ready!")

Training on  : 5000 examples
Validating on: 500 examples
✅ Trainer ready!


In [16]:
import torch

torch.cuda.empty_cache()

print("Starting LoRA fine-tuning...")
print("Expected time: ~30-40 mins on T4 GPU")
print("Watch the loss decrease each step!\n")

trainer.train()

print("\n Training complete!")

🚀 Starting LoRA fine-tuning...
Expected time: ~30-40 mins on T4 GPU
Watch the loss decrease each step!



Step,Training Loss,Validation Loss
100,0.699228,0.681440
200,0.672392,0.664681
300,0.679153,0.657308



✅ Training complete!


In [18]:
# Save LoRA adapter locally
model.save_pretrained("./medical-mistral-lora-final")
tokenizer.save_pretrained("./medical-mistral-lora-final")
print("Adapter saved locally!")

# Push to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(
    repo_id="medical-mistral-lora",
    exist_ok=True
)

model.push_to_hub("samboateng190/medical-mistral-lora")
tokenizer.push_to_hub("samboateng190/medical-mistral-lora")

print("Model pushed to HuggingFace Hub!")
print(" Live at: https://huggingface.co/samboateng190/medical-mistral-lora")

Adapter saved locally!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  73%|#######3  | 39.9MB / 54.6MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to HuggingFace Hub!
 Live at: https://huggingface.co/samboateng190/medical-mistral-lora


In [20]:
from transformers import BitsAndBytesConfig
import torch

# ── Inference helper ──────────────────────────────────────────
def generate_response(mdl, tok, question, max_new_tokens=200):
    prompt = f"""### Instruction:
Answer this question truthfully

### Input:
{question}

### Response:
"""
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tok.eos_token_id
        )
    response = tok.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# ── Test fine-tuned model only (base already in memory) ───────
test_questions = [
    "What is the mechanism of action of beta blockers?",
    "What are the symptoms of diabetic ketoacidosis?",
    "How does aspirin prevent blood clots?",
]

print("🟢 Testing FINE-TUNED model...\n")
for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    print(generate_response(model, tokenizer, q))

🟢 Testing FINE-TUNED model...


Q: What is the mechanism of action of beta blockers?
Beta blockers are a class of medications that work by blocking the action of beta-adrenergic receptors in the body. These receptors are found in various tissues, including the heart, blood vessels, and lungs, and are activated by the hormone adrenaline. By blocking the action of adrenaline on these receptors, beta blockers can help to reduce the heart rate, lower blood pressure, and improve blood flow to the heart and other organs. This can be particularly beneficial in the treatment of conditions such as hypertension, angina, and heart failure. However, it is important to note that beta blockers can have side effects and should only be used under the supervision of a healthcare provider.

Q: What are the symptoms of diabetic ketoacidosis?
The symptoms of diabetic ketoacidosis include nausea, vomiting, and abdominal pain.

Q: How does aspirin prevent blood clots?
Aspirin prevents blood clots by inhibit

In [21]:
# Write model card
model_card = """---
language: en
license: apache-2.0
tags:
- medical
- llm
- lora
- fine-tuned
- mistral
- peft
base_model: mistralai/Mistral-7B-Instruct-v0.3
---

# Medical QA LoRA — Fine-tuned Mistral 7B

Fine-tuned Mistral 7B on medical QA data using LoRA (Low-Rank Adaptation).
Trained as part of an ML Engineer portfolio project.

## Model Details

| Parameter | Value |
|-----------|-------|
| Base Model | Mistral-7B-Instruct-v0.3 |
| Fine-tuning Method | LoRA + 4-bit QLoRA |
| Dataset | Medical Meadow Flashcards (5,000 examples) |
| LoRA Rank | 16 |
| LoRA Alpha | 32 |
| Training Epochs | 1 |
| Final Val Loss | 0.6573 |
| Hardware | T4 GPU (Google Colab) |
| Training Time | ~56 minutes |

## Training Results

| Step | Train Loss | Val Loss |
|------|-----------|----------|
| 100  | 0.6992    | 0.6814   |
| 200  | 0.6724    | 0.6647   |
| 300  | 0.6792    | 0.6573   |

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
adapter    = "samboateng190/medical-mistral-lora"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map="auto")
model     = PeftModel.from_pretrained(model, adapter)

prompt = \"\"\"### Instruction:
Answer this question truthfully

### Input:
What is the mechanism of action of beta blockers?

### Response:
\"\"\"

inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Limitations
- Trained on flashcard-style QA — may not handle complex clinical reasoning
- Not a substitute for professional medical advice
- Should be validated before any clinical application
"""

with open("./medical-mistral-lora-final/README.md", "w") as f:
    f.write(model_card)

# Push model card to hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="./medical-mistral-lora-final/README.md",
    path_in_repo="README.md",
    repo_id="samboateng190/medical-mistral-lora",
)
print(" Model card pushed to HuggingFace Hub!")
print(" https://huggingface.co/samboateng190/medical-mistral-lora")

 Model card pushed to HuggingFace Hub!
 https://huggingface.co/samboateng190/medical-mistral-lora
